# UEBA Portable — Entraînement per-user sur données **normales**

Ce notebook apprend, **pour chaque utilisateur**, son comportement *normal* à
partir d'un dataset **propre** (sans incident connu). C'est le cœur de l'UEBA
de production : on fige une baseline par entité, puis on détecte plus tard les
écarts sur des données live (`ueba detect`).

> **Hypothèse** : le CSV fourni ne contient **que de l'activité normale**
> (les jours d'attaque/incident ont été retirés en amont). La séparation
> « données propres » relève de toi ; la classe apprend la normalité de ce
> qu'on lui donne.

## 0. Installation

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # --force-reinstall garantit de récupérer la DERNIÈRE version du code
    # (sinon Colab garde l'ancienne en cache). --no-deps = rapide.
    !pip install -q --upgrade --force-reinstall --no-deps "git+https://github.com/assia-xnz/ueba-portable.git"
    print("✅ Installé. Si tu avais déjà exécuté des cellules : Exécution → Redémarrer la session, puis relance.")
else:
    sys.path.insert(0, "../src")

print("Environnement :", "Google Colab" if IN_COLAB else "Local")

## 1. Upload du dataset NORMAL

Sur Colab, dépose ton export Wazuh **propre** (schéma natif
`data.win.eventdata.*`). En local, on retombe sur la fixture du dépôt.

In [ ]:
if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    CSV_PATH = next(iter(uploaded))
else:
    CSV_PATH = "../tests/integration/fixtures/sample_logs.csv"

print("Dataset :", CSV_PATH)

## 2. Normalisation via `WazuhAdapter`

In [ ]:
import csv
from pathlib import Path

from ueba.adapters.wazuh import WazuhAdapter

with Path(CSV_PATH).open(newline="", encoding="utf-8") as f:
    records = list(csv.DictReader(f))

events = WazuhAdapter().normalize(records)

dates = sorted({e.timestamp.date().isoformat() for e in events})
users = sorted({e.user for e in events})
print(f"Événements normalisés : {len(events)}")
if events:
    print(f"Période               : {dates[0]} → {dates[-1]} ({len(dates)} jours)")
    print(f"Utilisateurs          : {len(users)} → {users}")
else:
    print("⚠️  0 événement : vérifie le format du CSV (colonnes / timestamps).")

## 3. Extraction des features (fenêtres glissantes)

In [ ]:
from collections import Counter
from datetime import timedelta

from ueba.domain.features import UEBAFeatureExtractor

extractor = UEBAFeatureExtractor(
    window_size=timedelta(hours=1),
    window_step=timedelta(minutes=30),
)
vectors = extractor.extract(events)

per_user = Counter(v.user for v in vectors)
print(f"Vecteurs (utilisateur × fenêtre) : {len(vectors)}")
print("Fenêtres par utilisateur :")
for user, n in sorted(per_user.items(), key=lambda kv: -kv[1]):
    print(f"  {user:25s} {n:4d}")

## 4. Entraînement per-user sur **100 %** du normal

Un modèle dédié est appris par utilisateur, sur l'intégralité de ses fenêtres
(`train_ratio=1.0`). Un utilisateur avec moins de `min_windows_per_user`
fenêtres n'obtient pas de modèle (baseline jugée trop courte) — baisse ce
seuil si besoin.

In [ ]:
from ueba.domain.per_user_ensemble import PerUserAnomalyEnsemble

MIN_WINDOWS = 30  # baisse cette valeur si peu de fenêtres par utilisateur

model = PerUserAnomalyEnsemble(
    min_windows_per_user=MIN_WINDOWS,
    train_ratio=1.0,        # apprend sur TOUT le normal fourni
    svm_nu=0.05,            # baseline propre : on tolère peu d'outliers → moins de FP
    n_estimators=200,
    majority_threshold=2,
    random_state=42,
)
model.fit(vectors)

print(f"Modèles entraînés : {len(model.trained_users)} / {len(per_user)} utilisateurs")
print("Utilisateurs avec baseline :", model.trained_users)
not_trained = sorted(set(per_user) - set(model.trained_users))
if not_trained:
    print("Sans modèle (trop peu de fenêtres ou compte filtré) :", not_trained)

## 5. Contrôle de cohérence — le modèle reconnaît-il le normal ?

On rescore les **mêmes** données normales. Un bon modèle ne doit flaguer
qu'une **faible** proportion de fenêtres (de l'ordre du seuil du 95ᵉ
percentile, atténué par le vote majoritaire ≥ 2/3). Un taux élevé signale une
baseline trop courte ou un comportement réellement très variable pour cet
utilisateur.

In [ ]:
import pandas as pd

verdicts = model.predict(vectors)
df = pd.DataFrame(
    {
        "user": [v.user for v in vectors],
        "is_anomaly": [verd.is_anomaly for verd in verdicts],
        "was_in_training": [verd.was_in_training for verd in verdicts],
    }
)

trained_df = df[df["was_in_training"]]
if not trained_df.empty:
    overall = trained_df["is_anomaly"].mean()
    print(f"Taux d'anomalies sur le normal (utilisateurs entraînés) : {overall:.1%}")
    print("Par utilisateur :")
    rate = trained_df.groupby("user")["is_anomaly"].mean().sort_values(ascending=False)
    for user, r in rate.items():
        print(f"  {user:25s} {r:6.1%}")
else:
    print("Aucun utilisateur entraîné : baisse MIN_WINDOWS à la cellule 4.")

## 6. Visualisations

In [ ]:
import matplotlib.pyplot as plt

users_axis = sorted(per_user)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# (a) Fenêtres par utilisateur
counts = [per_user[u] for u in users_axis]
ax1.bar(users_axis, counts, color="#1f77b4")
ax1.set_title("Fenêtres par utilisateur")
ax1.set_ylabel("nombre de fenêtres")
ax1.tick_params(axis="x", rotation=45)

# (b) Taux d'anomalies sur le normal (utilisateurs entraînés)
if not trained_df.empty:
    rate = trained_df.groupby("user")["is_anomaly"].mean().reindex(
        sorted(model.trained_users)
    )
    ax2.bar(rate.index, rate.values, color="#2ca02c")
    ax2.set_title("Taux d'anomalies sur le normal (doit rester bas)")
    ax2.set_ylabel("part de fenêtres flaguées")
    ax2.tick_params(axis="x", rotation=45)
else:
    ax2.text(0.5, 0.5, "Aucun modèle entraîné\n(baisse MIN_WINDOWS)", ha="center", va="center")
    ax2.set_axis_off()

plt.tight_layout()
plt.show()

## 7. Sauvegarde et téléchargement du modèle de baseline

In [ ]:
MODEL_PATH = "per_user_baseline.joblib"
model.save(MODEL_PATH)

# Vérification : rechargement → état entraîné restauré + verdicts identiques.
reloaded = PerUserAnomalyEnsemble.load(MODEL_PATH)
assert reloaded.is_fitted
assert [v.is_anomaly for v in verdicts] == [v.is_anomaly for v in reloaded.predict(vectors)]
print(f"Modèle de baseline sauvegardé : {MODEL_PATH}")
print("→ À réutiliser pour détecter sur des données live (ueba detect).")

if IN_COLAB:
    from google.colab import files

    files.download(MODEL_PATH)